# 🏦 Finance RAG Agent

A small, self-contained **Retrieval-Augmented Generation (RAG) agent** for finance/risk questions, built with Claude's tool-calling API.

Given a question, the agent decides for itself — turn by turn — whether it needs to:

- 🔎 **search** an internal knowledge base (semantic vector search, not keyword matching),
- 🧮 **calculate** something (e.g. compound interest), or
- 📈 **fetch a live stock price**,

and then reasons over whatever tool results come back to produce a grounded final answer.

It's intentionally small — a single notebook, no vector DB, no framework (no LangChain/LlamaIndex) — so the full request/response loop is easy to read end-to-end. That also makes it easy to swap in your own documents, tools, or a real vector store later (see [Next Steps](#-next-steps--how-to-extend-this)).

> 💡 **Why this exists:** most "RAG demo" notebooks either fake retrieval with keyword lookup, or hide all the interesting logic inside a framework. This one shows real embedding-based retrieval *and* the actual agent loop, in plain Python, so you can see exactly what's happening at each step.

## Table of Contents
1. [Architecture](#-architecture)
2. [Setup](#1--install-dependencies)
3. [Knowledge Base](#2--knowledge-base-swap-this-for-your-own-docs--chunked-pdfs)
4. [Retrieval (real RAG)](#3--embeddings--vector-search-the-actual-rag-part)
5. [Tools + Anthropic Client](#4--anthropic-client--tools)
6. [Agent Loop](#5--the-agent-loop-tool-calling--react-style)
7. [Demo](#6--try-it)
8. [Next Steps](#-next-steps--how-to-extend-this)

## 🏗 Architecture

```
                     ┌─────────────────────┐
                     │   User question      │
                     └──────────┬───────────┘
                                ▼
                     ┌─────────────────────┐
                     │   Claude (agent)     │◄──────────────┐
                     │  decides which tool,  │               │
                     │  if any, to call      │               │
                     └──────────┬───────────┘               │
                                ▼                            │
              ┌─────────────────┼─────────────────┐          │
              ▼                 ▼                 ▼          │
   ┌───────────────┐  ┌────────────────┐  ┌───────────────┐  │
   │ knowledge_base │  │   calculator   │  │  stock_price   │  │
   │    _search     │  │  (safe eval)   │  │  (yfinance)    │  │
   │ (embeddings +  │  └────────────────┘  └───────────────┘  │
   │ cosine sim.)   │                                          │
   └───────┬────────┘                                          │
           │                    tool result fed back            │
           └──────────────────────────────────────────────────┘
                                ▼
                     ┌─────────────────────┐
                     │   Final answer       │
                     └─────────────────────┘
```

The retrieval step (top-left box) is real semantic search: questions and knowledge-base entries are both embedded with a sentence-transformer model, and the closest passages by cosine similarity are returned to Claude as tool results. Claude then writes the final answer grounded in that retrieved text.


## 1 — Install dependencies

In [ ]:
%pip install -q sentence-transformers anthropic yfinance numpy


## 2 — Knowledge base (swap this for your own docs / chunked PDFs)

For this demo the "corpus" is a small hardcoded dictionary of finance/risk topics. In a real project you'd
replace this cell with something that loads and chunks your own documents (10-Ks, policy manuals, research
notes, etc.) — the retrieval code in the next cell doesn't need to change at all, since it just operates on
a list of text chunks.

A typical swap-in looks like:

```python
# Example: load & chunk a PDF instead of using the hardcoded dict
# from pypdf import PdfReader
# reader = PdfReader("my_10k.pdf")
# full_text = "\n".join(page.extract_text() for page in reader.pages)
# docs = chunk_text(full_text, chunk_size=300)   # your own chunking function
# doc_keys = [f"chunk_{i}" for i in range(len(docs))]
```


In [ ]:
KNOWLEDGE_BASE = {
    "credit risk": (
        "Credit risk is the probability that a borrower will fail to repay a loan or meet "
        "contractual obligations. Its three components are Probability of Default (PD), "
        "Loss Given Default (LGD), and Exposure at Default (EAD). Banks hold regulatory "
        "capital proportional to their credit risk exposure under frameworks like Basel III."
    ),
    "model risk management": (
        "Model risk management (MRM) is the discipline of identifying, measuring, and "
        "controlling the risk that a model's errors or misuse lead to adverse business or "
        "regulatory outcomes. It covers model development, independent validation, ongoing "
        "monitoring, and governance \u2014 increasingly extended to AI/ML and generative AI models."
    ),
    "ai model validation": (
        "Validating an AI/ML model involves checking conceptual soundness, testing on "
        "out-of-sample and adversarial data, monitoring for drift, and evaluating fairness, "
        "robustness, and explainability, in addition to standard performance metrics like "
        "accuracy or AUC."
    ),
    "what is an etf": (
        "An ETF (Exchange-Traded Fund) is a basket of securities that trades on an exchange "
        "like a single stock, offering diversification at low cost and intraday liquidity."
    ),
    "compound interest": (
        "Compound interest is interest calculated on both the principal and the accumulated "
        "interest from prior periods: A = P(1 + r/n)^(nt)."
    ),
}

docs = list(KNOWLEDGE_BASE.values())
doc_keys = list(KNOWLEDGE_BASE.keys())
print(f"Loaded {len(docs)} knowledge base entries.")


## 3 — Embeddings + vector search (the actual RAG part)

This is real semantic search, not keyword or TF-IDF matching: both the knowledge base and the incoming
query are embedded with a local sentence-transformer model, and retrieval is done by cosine similarity
over those embeddings. `all-MiniLM-L6-v2` is used because it's small, fast, free, and runs locally after
the first download (no API calls needed for retrieval itself).


In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")  # small, fast, free, local after download

doc_embeddings = embedder.encode(docs, normalize_embeddings=True)


def retrieve(query, top_k=2):
    """Return the top_k most semantically similar knowledge base chunks to the query."""
    q_emb = embedder.encode([query], normalize_embeddings=True)[0]
    sims = doc_embeddings @ q_emb  # cosine similarity (vectors are already normalized)
    top_idx = np.argsort(sims)[::-1][:top_k]
    return [(doc_keys[i], docs[i], float(sims[i])) for i in top_idx]


# Sanity check: this query doesn't share any keywords with the "model risk management" entry,
# but should still retrieve it on meaning alone.
print("Query: \"how do regulators think about validating ML models?\"\n")
for key, text, score in retrieve("how do regulators think about validating ML models?"):
    print(f"[{score:.3f}] {key}")


## 4 — Anthropic client & tools

Set your API key as an environment variable (recommended) before running this cell:

```bash
export ANTHROPIC_API_KEY="sk-ant-..."
```

If it isn't already set, the cell below will prompt for it securely (via `getpass`) instead of asking you
to paste it into the notebook in plain text — **never commit an API key to git.** In Google Colab you can
also store it under `Secrets` (🔑 icon in the left sidebar) and load it with `google.colab.userdata`.

Three tools are defined for the agent:

| Tool | Purpose |
|---|---|
| `knowledge_base_search` | Semantic search over the internal knowledge base (Cell 3) |
| `calculator` | Safely evaluates arithmetic expressions (e.g. compound interest) — uses an AST whitelist, **not** `eval()` |
| `stock_price` | Fetches a live closing price via `yfinance` |


In [ ]:
import os
import getpass
import anthropic

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Enter your Anthropic API key: ")

client = anthropic.Anthropic()


def calculator(expression: str) -> str:
    """Safely evaluate a basic arithmetic expression (no eval() — AST whitelist only)."""
    import ast, operator
    ops = {ast.Add: operator.add, ast.Sub: operator.sub, ast.Mult: operator.mul,
           ast.Div: operator.truediv, ast.Pow: operator.pow, ast.USub: operator.neg}

    def _eval(node):
        if isinstance(node, ast.Constant):
            return node.value
        if isinstance(node, ast.BinOp):
            return ops[type(node.op)](_eval(node.left), _eval(node.right))
        if isinstance(node, ast.UnaryOp):
            return ops[type(node.op)](_eval(node.operand))
        raise ValueError("unsupported expression")

    return str(_eval(ast.parse(expression, mode="eval").body))


def stock_price(ticker: str) -> str:
    """Fetch the latest closing price for a ticker using yfinance."""
    import yfinance as yf
    hist = yf.Ticker(ticker).history(period="1d")
    if hist.empty:
        return f"No data found for {ticker}"
    price = hist["Close"].iloc[-1]
    return f"{ticker.upper()} last close: ${price:.2f}"


def knowledge_base_search(query: str) -> str:
    """Search the internal finance knowledge base and return the most relevant passages."""
    results = retrieve(query, top_k=2)
    return "\n\n".join(f"[{key}] {text}" for key, text, score in results)


TOOLS = [
    {
        "name": "knowledge_base_search",
        "description": "Search the internal finance/risk knowledge base for relevant background info.",
        "input_schema": {
            "type": "object",
            "properties": {"query": {"type": "string"}},
            "required": ["query"],
        },
    },
    {
        "name": "calculator",
        "description": "Evaluate a basic arithmetic expression, e.g. for compound interest calculations.",
        "input_schema": {
            "type": "object",
            "properties": {"expression": {"type": "string"}},
            "required": ["expression"],
        },
    },
    {
        "name": "stock_price",
        "description": "Get the latest closing price for a stock ticker symbol.",
        "input_schema": {
            "type": "object",
            "properties": {"ticker": {"type": "string"}},
            "required": ["ticker"],
        },
    },
]

TOOL_FUNCTIONS = {
    "knowledge_base_search": knowledge_base_search,
    "calculator": calculator,
    "stock_price": stock_price,
}


## 5 — The agent loop (tool-calling / ReAct-style)

Each turn, Claude either:
- returns a final text answer (loop ends), or
- requests one or more tool calls, which are executed locally and fed back in as `tool_result` blocks,
  after which Claude gets another turn with that new information.

This is the same request → tool call → tool result → next request cycle used by most production tool-using
agents, just written out explicitly instead of hidden behind a framework. Pass `verbose=True` to print each
tool call as it happens — handy for seeing *why* the agent answered the way it did.


In [ ]:
def run_agent(user_message, max_turns=4, model="claude-sonnet-4-5", verbose=False):
    messages = [{"role": "user", "content": user_message}]

    for turn in range(max_turns):
        response = client.messages.create(
            model=model,
            max_tokens=800,
            tools=TOOLS,
            messages=messages,
        )

        tool_calls = [b for b in response.content if b.type == "tool_use"]
        if not tool_calls:
            # No more tool calls -> final answer
            return "".join(b.text for b in response.content if b.type == "text")

        messages.append({"role": "assistant", "content": response.content})

        tool_results = []
        for call in tool_calls:
            if verbose:
                print(f"  \U0001F527 tool call: {call.name}({call.input})")
            fn = TOOL_FUNCTIONS[call.name]
            try:
                result = fn(**call.input)
            except Exception as e:
                result = f"Error running {call.name}: {e}"
            tool_results.append({
                "type": "tool_result",
                "tool_use_id": call.id,
                "content": str(result),
            })
        messages.append({"role": "user", "content": tool_results})

    return "Reached max turns without a final answer."


## 6 — Try it

In [ ]:
test_questions = [
    "What is model risk management and how does it relate to AI validation?",
    "What's the current price of AAPL stock?",
    "If I invest $10000 at 5% annual interest compounded yearly for 3 years, roughly how much do I have? Use (1.05)**3 * 10000",
]

for q in test_questions:
    print(f"\n\U0001F64B {q}")
    print(f"\U0001F916 {run_agent(q, verbose=True)}")
    print("-" * 70)


## Summary — what this demonstrates

| Component | Naive approach | This notebook |
|---|---|---|
| Matching | Exact keyword / dictionary lookup | Sentence-transformer embeddings (`all-MiniLM-L6-v2`) — real semantic search |
| Generation | Hardcoded string return | Claude generates the answer, grounded in retrieved passages |
| Behavior | Fixed responses only | **Agent**: decides per-query whether to search, calculate, or fetch live data, then reasons over the result |
| Extensibility | Add dict entries manually | Add any tool (a function + JSON schema) and the agent can use it |

## 🚀 Next steps / how to extend this

- **Real documents** — chunk and load PDFs (10-Ks, policy docs, research notes) into the knowledge base instead of the hardcoded dict.
- **A real vector store** — swap the in-memory `doc_embeddings` array for FAISS, Chroma, or pgvector once the corpus grows beyond a few hundred chunks.
- **Evaluation** — log retrieved-chunk relevance and tool-choice accuracy against a small labeled test set.
- **More tools** — SEC filing lookups (EDGAR API), FRED economic data, portfolio math, etc.
- **Streaming / UI** — wrap `run_agent` in a small Streamlit or Gradio app for an interactive demo.

---
*Built as a portfolio project to demonstrate RAG + tool-using agents with Claude's API. See the `README.md` in this repo for setup instructions.*
